### This ipynb is for public score 0.94857

### Import Dependencies

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm
import networkx as nx

### 1. Data Loading & Sanity Checks

In [2]:
# Replace DATA_DIR with your own directory where the data is stored
DATA_DIRECTORY = "C:\\National University of Singapore\\OneDrive - National University of Singapore\\NUS\\Year 4 Semester 1\\BT4012\\Kaggle Competition\\BT4012KaggleCompetitionDataset2026"

test_data_filepath = os.path.join(DATA_DIRECTORY, "test.csv")
train_data_filepath = os.path.join(DATA_DIRECTORY, "train.csv")
sample_submission_filepath = os.path.join(DATA_DIRECTORY, "sample_submission.csv")
txs_edgelist_filepath = os.path.join(DATA_DIRECTORY, "txs_edgelist.csv")

print("***** Step 1 *****")
print("Loading datasets...")

test_df = pd.read_csv(test_data_filepath)
train_df = pd.read_csv(train_data_filepath)
sample_submission_df = pd.read_csv(sample_submission_filepath)
edgelist_df = pd.read_csv(txs_edgelist_filepath)

print(f"\ntest.csv: {test_df.shape[0]} rows and {test_df.shape[1]} columns")
print(f"train.csv: {train_df.shape[0]} rows and {train_df.shape[1]} columns")
print(f"sample_submission.csv: {sample_submission_df.shape[0]} rows and {sample_submission_df.shape[1]} columns")
print(f"txs_edgelist.csv: {edgelist_df.shape[0]} rows and {edgelist_df.shape[1]} columns")

# Label Distribution
print("\nTrain data label distribution")
print(train_df["label"].value_counts(dropna=False))

# Time step ranges
print("\nTime step ranges")
print(f"Train data time_step: {train_df["time_step"].min()} to {train_df["time_step"].max()}")
print(f"Test data time_step: {test_df["time_step"].min()} to {test_df["time_step"].max()}")

# Sanity check to confirm test set occurs after train set
assert test_df["time_step"].min() > train_df["time_step"].max(), "Test set time steps overlap with train set"
print("\nConfirmed test set time steps to be after train set time steps")

# Missing values
train_set_missing_values = train_df.isnull().sum().sum()
test_set_missing_values = test_df.isnull().sum().sum()
print("\nMissing values")
print(f"Train set missing values: {train_set_missing_values}")
print(f"Test set missing values: {test_set_missing_values}")

# Sample submission format
print("\nSample submission format")
print(sample_submission_df.head())
print(f"\nColumns: {list(sample_submission_df.columns)}")

# Edge list sample
print("\nEdge list sample")
print(edgelist_df.head())
print(f"Unique source nodes (txId1): {edgelist_df["txId1"].nunique():,}")
print(f"Unique target nodes (txId2): {edgelist_df["txId2"].nunique():,}")

print("\nStep 1 completed")
#print(test_data_filepath)
#print(train_data_filepath)
#print(sample_submission_filepath)
#print(txs_edgelist_filepath)

***** Step 1 *****
Loading datasets...

test.csv: 15329 rows and 168 columns
train.csv: 141772 rows and 168 columns
sample_submission.csv: 15329 rows and 2 columns
txs_edgelist.csv: 234355 rows and 2 columns

Train data label distribution
label
NaN    110537
0.0     27591
1.0      3644
Name: count, dtype: int64

Time step ranges
Train data time_step: 1 to 35
Test data time_step: 36 to 49

Confirmed test set time steps to be after train set time steps

Missing values
Train set missing values: 110537
Test set missing values: 0

Sample submission format
   index  target
0      0     0.5
1      1     0.5
2      2     0.5
3      3     0.5
4      4     0.5

Columns: ['index', 'target']

Edge list sample
       txId1      txId2
0  230425980    5530458
1  232022460  232438397
2  230460314  230459870
3  230333930  230595899
4  232013274  232029206
Unique source nodes (txId1): 166,345
Unique target nodes (txId2): 148,447

Step 1 completed


### 2. Exploratory Data Analysis

In [3]:
print("Step 2: Performing EDA")

# Identify feature columns

feature_columns = [column for column in train_df.columns if column.startswith("feat_")]
print(f"\nFeature columns count: {len(feature_columns)}, from {feature_columns[0]} to {feature_columns[-1]}")

# Check temporal distribution of labels (licit vs illicit vs unknown per time_step)
print("\nLabel distribution across time steps (first 5 and last 5)")
step_summary = pd.DataFrame({
    "total_txs": train_df.groupby("time_step").size(),
    "licit": train_df[train_df["label"].isin([0, 0.0, "0"])].groupby("time_step").size(),
    "illicit": train_df[train_df["label"].isin([1, 1.0, "1"])].groupby("time_step").size(),
    "unlabeled": train_df[train_df["label"].isna()].groupby("time_step").size(),
}).fillna(0).astype(int)

step_summary["labeled_total"] = step_summary["licit"] + step_summary["illicit"]
step_summary["illicit_pct"] = (step_summary["illicit"] / step_summary["labeled_total"] * 100).round(2)

print("\nStep summary table of first and last 5 rows")
print(pd.concat([step_summary.head(5), step_summary.tail(5)]))

# Summary statistics across features
print("\nFeature stats sample (feat_1 to feat_5)")
print(train_df[feature_columns[:5]].describe().round(4))

print("\nStep 2 completed")

Step 2: Performing EDA

Feature columns count: 165, from feat_1 to feat_165

Label distribution across time steps (first 5 and last 5)

Step summary table of first and last 5 rows
           total_txs  licit  illicit  unlabeled  labeled_total  illicit_pct
time_step                                                                  
1               7880   2130       17       5733           2147         0.79
2               4544   1099       18       3427           1117         1.61
3               6621   1268       11       5342           1279         0.86
4               5693   1410       30       4253           1440         2.08
5               6803   1874        8       4921           1882         0.43
31              2816    604      106       2106            710        14.93
32              4525    981      342       3202           1323        25.85
33              3151    418       23       2710            441         5.22
34              2486    478       37       1971            5

### 3. Data Pre-Processing

In [ ]:
print(f"\n{'=' * 60}")
print("STEP 3: Data Preprocessing...")
print(f"{'=' * 60}")

# 1. Guarantee submission row alignment for test set
test_df = test_df.assign(original_order=np.arange(len(test_df)))
print(f"\nTagged test_df with 'original_order' (0 to {len(test_df) - 1})")

# 2. Detrend time-proxy features within each time_step to mitigate out-of-distribution drift
DRIFT_COLS = ["feat_100", "feat_101", "feat_103", "feat_136", "feat_139", "feat_137"]
detrended_feature_cols = [f"{c}_detrended" for c in DRIFT_COLS]

# Drop existing detrended columns if re-running cell
train_df = train_df.drop(columns=[c for c in detrended_feature_cols if c in train_df.columns])
test_df = test_df.drop(columns=[c for c in detrended_feature_cols if c in test_df.columns])

all_drift = pd.concat([train_df[["time_step"] + DRIFT_COLS], test_df[["time_step"] + DRIFT_COLS]], ignore_index=True)
medians = all_drift.groupby("time_step")[DRIFT_COLS].transform("median")
detrended_df = all_drift[DRIFT_COLS] - medians
detrended_df.columns = detrended_feature_cols

train_df = pd.concat([train_df, detrended_df.iloc[:len(train_df)]], axis=1)
test_df = pd.concat([test_df, detrended_df.iloc[len(train_df):].reset_index(drop=True)], axis=1)
print(f"Computed {len(detrended_feature_cols)} detrended features: {detrended_feature_cols}")

# 3. Dynamic Time-Step Z-Score Normalization for Top Macro-Sensitive Signals
TOP_Z_COLS = ["feat_1", "feat_2", "feat_5", "feat_41", "feat_47", "feat_53"]
zscore_feature_cols = [f"{c}_zscore" for c in TOP_Z_COLS]

# Drop existing zscore columns if re-running cell
train_df = train_df.drop(columns=[c for c in zscore_feature_cols if c in train_df.columns])
test_df = test_df.drop(columns=[c for c in zscore_feature_cols if c in test_df.columns])

all_raw_z = pd.concat([train_df[["time_step"] + TOP_Z_COLS], test_df[["time_step"] + TOP_Z_COLS]], ignore_index=True)
step_means = all_raw_z.groupby("time_step")[TOP_Z_COLS].transform("mean")
step_stds = all_raw_z.groupby("time_step")[TOP_Z_COLS].transform("std") + 1e-5
zscore_df = (all_raw_z[TOP_Z_COLS] - step_means) / step_stds
zscore_df.columns = zscore_feature_cols

train_df = pd.concat([train_df, zscore_df.iloc[:len(train_df)]], axis=1)
test_df = pd.concat([test_df, zscore_df.iloc[len(train_df):].reset_index(drop=True)], axis=1)
print(f"Computed {len(zscore_feature_cols)} dynamic time-step Z-scores: {zscore_feature_cols}")

# 4. Extract labeled subset for supervised training
labeled_mask = train_df["label"].isin([0, 1, 0.0, 1.0])
labeled_train_df = train_df[labeled_mask].copy()

# 5. Cast label to integer target (0 = licit, 1 = illicit)
labeled_train_df["target"] = labeled_train_df["label"].astype(int)

print(f"\nOriginal train rows : {len(train_df):,}")
print(f"Labeled train rows  : {len(labeled_train_df):,} ({len(labeled_train_df)/len(train_df)*100:.2f}%)")
print(f"Unlabeled train rows: {(~labeled_mask).sum():,} ({(~labeled_mask).sum()/len(train_df)*100:.2f}%)")

print(f"\nTarget class balance in labeled subset:")
print(labeled_train_df["target"].value_counts(normalize=False))
print(f"Illicit proportion: {labeled_train_df['target'].mean() * 100:.2f}%")

print("STEP 3 COMPLETE ✓")


STEP 3: Data Preprocessing...

Tagged test_df with 'original_order' (0 to 15328)
Computed 6 detrended features: ['feat_100_detrended', 'feat_101_detrended', 'feat_103_detrended', 'feat_136_detrended', 'feat_139_detrended', 'feat_137_detrended']
Computed 6 dynamic time-step Z-scores: ['feat_1_zscore', 'feat_2_zscore', 'feat_5_zscore', 'feat_41_zscore', 'feat_47_zscore', 'feat_53_zscore']


C:\Users\jerry\AppData\Local\Temp\ipykernel_37208\533381608.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df = test_df.assign(original_order=np.arange(len(test_df)))



Original train rows : 141,772
Labeled train rows  : 31,235 (22.03%)
Unlabeled train rows: 110,537 (77.97%)

Target class balance in labeled subset:
target
0    27591
1     3644
Name: count, dtype: int64
Illicit proportion: 11.67%
STEP 3 COMPLETE ✓


# DO NOT RUN THIS CELL

print("Step 4: Performing Graph Feature Engineering")

A node's in-degree measures its predecessors, and its out-degree measures its successors.
txId1 is the predecessor to txId2 (the successor).

1. Compute in-degree (predecessors: txId2 receives from txId1)
    and out-degree (successors: txId1 sends to txId2)
 Note: Calculated over the entire edge list, preserving all unlabeled hops.
in_degrees = edgelist_df.groupby("txId2").size().rename("graph_in_degree")
out_degrees = edgelist_df.groupby("txId1").size().rename("graph_out_degree")

print(f"\nUnique transactions with in-edges : {len(in_degrees):,}")
print(f"Unique transactions with out-edges: {len(out_degrees):,}")

2. Merge graph degree metrics into both labeled_train_df and test_df
def add_graph_features(df):
     Compute degree columns as aligned series to avoid fragmented in-place insertions
    in_deg = df["txId"].map(in_degrees).fillna(0).astype(int)
    out_deg = df["txId"].map(out_degrees).fillna(0).astype(int)
    deg_ratio = (in_deg + 1) / (out_deg + 1)
    net_flow = out_deg - in_deg
    
    Construct new features DataFrame and assign all at once
    graph_cols_df = pd.DataFrame({
        "graph_in_degree": in_deg,
        "graph_out_degree": out_deg,
        "graph_degree_ratio": deg_ratio,
        "graph_net_flow": net_flow,
    }, index=df.index)
    
    Concat horizontally and copy to ensure a single contiguous memory block
    return pd.concat([df, graph_cols_df], axis=1).copy()


print("\nMerging graph features into labeled_train_df and test_df")
labeled_train_df = add_graph_features(labeled_train_df)
test_df = add_graph_features(test_df)

graph_feature_cols = ["graph_in_degree", "graph_out_degree", "graph_degree_ratio", "graph_net_flow"]
print(f"\nAdded {len(graph_feature_cols)} graph features: {graph_feature_cols}")

Sanity check on the new features
print("\nGraph Features Summary (Labeled Train)")
print(labeled_train_df[graph_feature_cols].describe().round(4))

print("\nGraph Degrees by Label (Licit vs Illicit)")
print(labeled_train_df.groupby("target")[["graph_in_degree", "graph_out_degree"]].mean().round(4))

print("Step 4 completed")

### 4. Graph Feature Engineering Deeper Topologies

In [5]:
print("STEP 4: Graph Feature Engineering (Deeper Topologies)...")

# 1. 1-Hop Degree Metrics (Calculated over full graph across all time steps)
in_degrees = edgelist_df.groupby("txId2").size().rename("graph_in_degree")
out_degrees = edgelist_df.groupby("txId1").size().rename("graph_out_degree")

print(f"\nUnique transactions with in-edges : {len(in_degrees):,}")
print(f"Unique transactions with out-edges: {len(out_degrees):,}")

# 2. 2-Hop Degree Aggregations (Predecessor & Successor Flow Patterns)
print("Computing 2-hop predecessor and successor degree flows...")
edges_deg = edgelist_df.copy()
edges_deg["src_in_deg"] = edges_deg["txId1"].map(in_degrees).fillna(0)
edges_deg["src_out_deg"] = edges_deg["txId1"].map(out_degrees).fillna(0)
edges_deg["dst_in_deg"] = edges_deg["txId2"].map(in_degrees).fillna(0)
edges_deg["dst_out_deg"] = edges_deg["txId2"].map(out_degrees).fillna(0)

# Predecessors of txId2 (who funded this transaction?)
pred_stats = edges_deg.groupby("txId2").agg(
    pred_in_deg_mean=("src_in_deg", "mean"),
    pred_in_deg_max=("src_in_deg", "max"),
    pred_out_deg_mean=("src_out_deg", "mean"),
    pred_out_deg_max=("src_out_deg", "max"),
)

# Successors of txId1 (where did funds from this transaction go?)
succ_stats = edges_deg.groupby("txId1").agg(
    succ_in_deg_mean=("dst_in_deg", "mean"),
    succ_in_deg_max=("dst_in_deg", "max"),
    succ_out_deg_mean=("dst_out_deg", "mean"),
    succ_out_deg_max=("dst_out_deg", "max"),
)

# 3. Global Graph Topology: PageRank and Weakly Connected Components (WCC)
print("Computing PageRank and Weakly Connected Components...")
G = nx.from_pandas_edgelist(edgelist_df, source="txId1", target="txId2", create_using=nx.DiGraph())

pr_dict = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-5)
pr_series = pd.Series(pr_dict, name="graph_pagerank")

wcc_map = {}
for comp in nx.weakly_connected_components(G):
    c_size = len(comp)
    for node in comp:
        wcc_map[node] = c_size
wcc_series = pd.Series(wcc_map, name="graph_wcc_size")

graph_feature_cols = [
    "graph_in_degree", "graph_out_degree", "graph_degree_ratio", "graph_net_flow",
    "graph_pagerank", "graph_wcc_size",
    "pred_in_deg_mean", "pred_in_deg_max", "pred_out_deg_mean", "pred_out_deg_max",
    "succ_in_deg_mean", "succ_in_deg_max", "succ_out_deg_mean", "succ_out_deg_max"
]

# 4. Merge all 14 graph metrics cleanly into labeled_train_df and test_df
def add_graph_features(df):
    # If graph features were already added in a previous run, drop them to prevent duplicates
    existing_cols = [c for c in df.columns if c in graph_feature_cols]
    if existing_cols:
        df = df.drop(columns=existing_cols)
    
    in_deg = df["txId"].map(in_degrees).fillna(0).astype(int)
    out_deg = df["txId"].map(out_degrees).fillna(0).astype(int)
    deg_ratio = (in_deg + 1) / (out_deg + 1)
    net_flow = out_deg - in_deg
    pr = df["txId"].map(pr_series).fillna(0.0)
    wcc = df["txId"].map(wcc_series).fillna(1).astype(int)
    
    # 1-hop + global features
    base_graph_df = pd.DataFrame({
        "graph_in_degree": in_deg,
        "graph_out_degree": out_deg,
        "graph_degree_ratio": deg_ratio,
        "graph_net_flow": net_flow,
        "graph_pagerank": pr,
        "graph_wcc_size": wcc,
    }, index=df.index)
    
    # Merge 2-hop predecessor & successor stats
    df_merged = pd.concat([df, base_graph_df], axis=1)
    df_merged = df_merged.merge(pred_stats, left_on="txId", right_index=True, how="left")
    df_merged = df_merged.merge(succ_stats, left_on="txId", right_index=True, how="left")
    
    # Fill isolated/leaf nodes (no 2-hop edges) with 0
    stat_cols = list(pred_stats.columns) + list(succ_stats.columns)
    df_merged[stat_cols] = df_merged[stat_cols].fillna(0.0)
    
    # Guard against any duplicate column names from notebook re-runs
    df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]
    return df_merged.copy()

print("\nMerging 14 graph features into labeled_train_df and test_df...")
labeled_train_df = add_graph_features(labeled_train_df)
test_df = add_graph_features(test_df)
print(f"\nAdded {len(graph_feature_cols)} graph features: {graph_feature_cols}")

# Quick sanity check on the new features
print("\n--- Graph Features Summary (Labeled Train) ---")
print(labeled_train_df[graph_feature_cols].describe().round(4))

print("\n--- Graph Features by Target (Licit vs Illicit Means) ---")
print(labeled_train_df.groupby("target")[graph_feature_cols].mean().round(4).T)

# 5. Neighbor Ego-Network Features (Aggregating top signals across in/out edges + net flow diff)
print("\nComputing 1-hop neighbor ego-network feature aggregations...")
TOP_SIGNALS = ['feat_1', 'feat_41', 'feat_47', 'feat_53', 'feat_2', 'feat_5']
all_nodes_features = pd.concat([
    train_df[['txId'] + TOP_SIGNALS], 
    test_df[['txId'] + TOP_SIGNALS]
]).drop_duplicates('txId').set_index('txId')

edges_feats = edgelist_df.merge(all_nodes_features, left_on='txId1', right_index=True)
pred_feats = edges_feats.groupby('txId2')[TOP_SIGNALS].agg(['mean', 'max'])
pred_feats.columns = [f"pred_{col}_{stat}" for col, stat in pred_feats.columns]

edges_feats2 = edgelist_df.merge(all_nodes_features, left_on='txId2', right_index=True)
succ_feats = edges_feats2.groupby('txId1')[TOP_SIGNALS].agg(['mean', 'max'])
succ_feats.columns = [f"succ_{col}_{stat}" for col, stat in succ_feats.columns]

nbr_stats = pred_feats.join(succ_feats, how='outer').fillna(0.0)
neighbor_feature_cols = nbr_stats.columns.tolist()

# Merge neighbor stats into labeled_train_df and test_df
existing_nbr_cols = [c for c in labeled_train_df.columns if c in neighbor_feature_cols or c == 'flow_amount_diff']
if existing_nbr_cols:
    labeled_train_df = labeled_train_df.drop(columns=existing_nbr_cols)
    test_df = test_df.drop(columns=[c for c in test_df.columns if c in existing_nbr_cols])

labeled_train_df = labeled_train_df.merge(nbr_stats, left_on='txId', right_index=True, how='left')
test_df = test_df.merge(nbr_stats, left_on='txId', right_index=True, how='left')
labeled_train_df[neighbor_feature_cols] = labeled_train_df[neighbor_feature_cols].fillna(0.0)
test_df[neighbor_feature_cols] = test_df[neighbor_feature_cols].fillna(0.0)

# Flow differential: transaction amount minus mean amount of inputs
labeled_train_df['flow_amount_diff'] = labeled_train_df['feat_1'] - labeled_train_df['pred_feat_1_mean']
test_df['flow_amount_diff'] = test_df['feat_1'] - test_df['pred_feat_1_mean']
neighbor_feature_cols.append('flow_amount_diff')

print(f"Added {len(neighbor_feature_cols)} neighbor aggregation features: {neighbor_feature_cols}")

print("STEP 4 COMPLETE ✓")

STEP 4: Graph Feature Engineering (Deeper Topologies)...

Unique transactions with in-edges : 148,447
Unique transactions with out-edges: 166,345
Computing 2-hop predecessor and successor degree flows...
Computing PageRank and Weakly Connected Components...

Merging 14 graph features into labeled_train_df and test_df...

Added 14 graph features: ['graph_in_degree', 'graph_out_degree', 'graph_degree_ratio', 'graph_net_flow', 'graph_pagerank', 'graph_wcc_size', 'pred_in_deg_mean', 'pred_in_deg_max', 'pred_out_deg_mean', 'pred_out_deg_max', 'succ_in_deg_mean', 'succ_in_deg_max', 'succ_out_deg_mean', 'succ_out_deg_max']

--- Graph Features Summary (Labeled Train) ---
       graph_in_degree  graph_out_degree  graph_degree_ratio  graph_net_flow  \
count       31235.0000        31235.0000          31235.0000      31235.0000   
mean            1.7632            1.1024              1.8570         -0.6607   
std             6.2582            3.5615              5.7703          7.2951   
min     

### 5. Temporal Train / Validation Split

In [6]:
print(f"\n{'=' * 60}")
print("STEP 5: Temporal Train/Validation Split...")
print(f"{'=' * 60}")

# 1. Define model feature columns as an explicit whitelist:
#    165 raw features + 6 detrended + 6 time z-scores + 14 graph topologies + 25 neighbor ego-network = 216 features.
model_feature_cols = feature_columns + detrended_feature_cols + zscore_feature_cols + graph_feature_cols + neighbor_feature_cols
print(f"\nModel feature count: {len(model_feature_cols)} "
      f"({len(feature_columns)} raw + {len(detrended_feature_cols)} detrended + {len(zscore_feature_cols)} time z-scores + {len(graph_feature_cols)} graph + {len(neighbor_feature_cols)} neighbor)")

# Every model feature must exist in the test set too
missing_in_test = [c for c in model_feature_cols if c not in test_df.columns]
assert not missing_in_test, f"Features missing from test_df: {missing_in_test}"

# 2. Split by time_step: roughly the earliest 80% of labeled rows train,
#    the latest 20% validate. Never shuffle — test is strictly in the future.
split_step = labeled_train_df["time_step"].quantile(0.8)
trn_mask = labeled_train_df["time_step"] <= split_step
val_mask = labeled_train_df["time_step"] > split_step

X_train = labeled_train_df.loc[trn_mask, model_feature_cols]
y_train = labeled_train_df.loc[trn_mask, "target"]
X_val = labeled_train_df.loc[val_mask, model_feature_cols]
y_val = labeled_train_df.loc[val_mask, "target"]

trn_steps = labeled_train_df.loc[trn_mask, "time_step"]
val_steps = labeled_train_df.loc[val_mask, "time_step"]
print(f"\nSplit step (80% quantile of labeled time_step): {split_step}")
print(f"Train: time_step {trn_steps.min()}-{trn_steps.max()} | "
      f"{len(X_train):,} rows ({len(X_train)/len(labeled_train_df)*100:.1f}%) | "
      f"illicit {int(y_train.sum()):,} ({y_train.mean()*100:.2f}%)")
print(f"Val  : time_step {val_steps.min()}-{val_steps.max()} | "
      f"{len(X_val):,} rows ({len(X_val)/len(labeled_train_df)*100:.1f}%) | "
      f"illicit {int(y_val.sum()):,} ({y_val.mean()*100:.2f}%)")

# Sanity: no time step appears in both folds
assert trn_steps.max() < val_steps.min(), "Train and validation time steps overlap!"
print("✓ Confirmed: no time-step overlap between train and validation.")

print(f"\n{'=' * 60}")
print("STEP 5 COMPLETE ✓")
print(f"{'=' * 60}")


STEP 5: Temporal Train/Validation Split...

Model feature count: 216 (165 raw + 6 detrended + 6 time z-scores + 14 graph + 25 neighbor)

Split step (80% quantile of labeled time_step): 28.0
Train: time_step 1-28 | 25,207 rows (80.7%) | illicit 2,542 (10.08%)
Val  : time_step 29-35 | 6,028 rows (19.3%) | illicit 1,102 (18.28%)
✓ Confirmed: no time-step overlap between train and validation.

STEP 5 COMPLETE ✓


In [7]:
len(model_feature_cols)

216

### 6. Baseline Model Training (LightGBM)

Training steps 1 to 28

In [8]:
print("Step 6: Performing Baseline LightGBM Training")

SEED = 42

# 1. Class imbalance weight, computed on the TRAINING fold only
#    (using validation labels here would leak information).
n_licit = int((y_train == 0).sum())
n_illicit = int((y_train == 1).sum())
scale_pos_weight = n_licit / n_illicit
print(f"\nTrain fold: licit={n_licit:,}, illicit={n_illicit:,} "
      f"-> scale_pos_weight = {scale_pos_weight:.4f}")

# 2. Model definition
#    - metric="auc": the competition metric, and the only metric early stopping watches
#    - subsample_freq=1: required, otherwise LightGBM silently ignores `subsample`
#    - deterministic + force_row_wise: reproducible results across runs
model = lightgbm.LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    deterministic=True,
    force_row_wise=True,
    verbose=-1,
)

# 3. Fit on steps 1-28, monitor AUC on steps 29-35.
#    Early stopping halts when validation AUC hasn't improved for 100 rounds.
#    Caveat: because the validation set also picks the stopping round,
#    the validation AUC reported in Step 7 is slightly optimistic.
print("\nTraining (validation AUC logged every 100 rounds)...")
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lightgbm.early_stopping(stopping_rounds=100, verbose=True),
        lightgbm.log_evaluation(period=100),
    ],
)

# 4. Record the best round (reused in Step 8 if we refit on all labeled data)
best_iteration = model.best_iteration_
best_val_auc = model.best_score_["valid_0"]["auc"]
print(f"\nBest iteration : {best_iteration}")
print(f"Best val AUC   : {best_val_auc:.5f}")

# 5. Which features does the model rely on most? (gain = total loss reduction)
importance = pd.Series(
    model.booster_.feature_importance(importance_type="gain"),
    index=model_feature_cols,
).sort_values(ascending=False)
print("\n--- Top 15 Features by Gain ---")
print((importance.head(15) / importance.sum() * 100).round(2).rename("gain_%"))
print("\nGraph feature ranks (out of {}):".format(len(model_feature_cols)))
ranks = importance.rank(ascending=False).astype(int)
print(ranks[graph_feature_cols].sort_values())

print("Step 6 completed")

Step 6: Performing Baseline LightGBM Training

Train fold: licit=22,665, illicit=2,542 -> scale_pos_weight = 8.9162

Training (validation AUC logged every 100 rounds)...


c:\Users\jerry\anaconda3\envs\BT4012\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.989449
[200]	valid_0's auc: 0.994422
[300]	valid_0's auc: 0.99622
[400]	valid_0's auc: 0.996746
[500]	valid_0's auc: 0.997066
[600]	valid_0's auc: 0.997291
[700]	valid_0's auc: 0.997454
[800]	valid_0's auc: 0.997562
[900]	valid_0's auc: 0.997632
[1000]	valid_0's auc: 0.997707
[1100]	valid_0's auc: 0.997753
[1200]	valid_0's auc: 0.997768
Did not meet early stopping. Best iteration is:
[1142]	valid_0's auc: 0.997789

Best iteration : 1142
Best val AUC   : 0.99779

--- Top 15 Features by Gain ---
feat_41               16.85
feat_53               13.54
feat_47               13.37
feat_2                 6.94
feat_5                 5.58
feat_136_detrended     3.16
feat_101               3.04
feat_58                2.64
feat_59                2.60
feat_90                1.85
feat_100               1.51
feat_132               1.28
feat_49                1.12
feat_106               0.93
feat_161               0

### 7. Validation and Metrics

After training on steps 1 to 28 previously, now we perform validation which tests against steps 29 to 35

In [9]:
print("Step 7: Performing validation evaluation & metrics")

# 1. Predict continuous probabilities on the validation fold (steps 29-35)
val_probs = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_probs)

print(f"\nOffline validation ROC-AUC: {val_auc:.5f}")

# 2. Evaluate performance across different decision thresholds (AML trade-off analysis)
#    Lower threshold -> higher Recall (catches more fraud, more false alarms)
#    Higher threshold -> higher Precision (fewer false alarms, misses subtle fraud)
for threshold in [0.3, 0.5, 0.7]:
    preds_binary = (val_probs >= threshold).astype(int)
    print(f"\nClassification metrics @ decision threshold = {threshold}")
    print(classification_report(y_val, preds_binary, digits=4, target_names=["Licit (0)", "Illicit (1)"]))

print("Step 7 completed")

Step 7: Performing validation evaluation & metrics

Offline validation ROC-AUC: 0.99779

Classification metrics @ decision threshold = 0.3
              precision    recall  f1-score   support

   Licit (0)     0.9885    0.9951    0.9918      4926
 Illicit (1)     0.9775    0.9483    0.9627      1102

    accuracy                         0.9866      6028
   macro avg     0.9830    0.9717    0.9772      6028
weighted avg     0.9865    0.9866    0.9865      6028


Classification metrics @ decision threshold = 0.5
              precision    recall  f1-score   support

   Licit (0)     0.9850    0.9968    0.9908      4926
 Illicit (1)     0.9847    0.9319    0.9576      1102

    accuracy                         0.9849      6028
   macro avg     0.9848    0.9643    0.9742      6028
weighted avg     0.9849    0.9849    0.9847      6028


Classification metrics @ decision threshold = 0.7
              precision    recall  f1-score   support

   Licit (0)     0.9783    0.9980    0.9880      4

### 8. (NEW) Retraining on Full Labeled Data (Steps 1 to 35) & Generate Submission

In [ ]:
print("STEP 8: Retraining on Full Labeled Data & Generating Submission...")

# 1. Prepare full labeled dataset (Steps 1 to 35)
X_full = labeled_train_df[model_feature_cols]
y_full = labeled_train_df["target"]

n_licit_full = int((y_full == 0).sum())
n_illicit_full = int((y_full == 1).sum())
scale_pos_weight_full = n_licit_full / n_illicit_full

print(f"\nFull Labeled Data (Steps 1-35):")
print(f"  Rows          : {len(X_full):,}")
print(f"  Licit (0)     : {n_licit_full:,}")
print(f"  Illicit (1)   : {n_illicit_full:,}")
print(f"  Scale Weight  : {scale_pos_weight_full:.4f}")
print(f"  Trees (rounds): {best_iteration} (optimal round count from Step 6)")

# 2. Train final production ensemble on 100% of labeled data (Dual-Architecture 10-Model Ensemble)
SEEDS = [42, 123, 456, 789, 2026]
print(f"\nTraining Dual-Architecture 10-Model Ensemble on steps 1-35 (5 seeds x 2 architectures)...")

# 3. Ensure test_df is strictly in its original order
test_df = test_df.sort_values("original_order").reset_index(drop=True)

test_preds_list = []
# Architecture A: Standard Depth 6, lr 0.03
for i, seed in enumerate(SEEDS, 1):
    print(f"  [Architecture A - Seed {seed}] Training model {i}/10...")
    m_a = lightgbm.LGBMClassifier(
        objective="binary",
        metric="auc",
        n_estimators=best_iteration,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight_full,
        random_state=seed,
        deterministic=True,
        force_row_wise=True,
        verbose=-1,
    )
    m_a.fit(X_full, y_full)
    test_preds_list.append(m_a.predict_proba(test_df[model_feature_cols])[:, 1])

# Architecture B: Deeper Depth 7, lr 0.02
best_iteration_b = int(best_iteration * 1.25)
for i, seed in enumerate(SEEDS, 1):
    print(f"  [Architecture B - Seed {seed}] Training model {i+len(SEEDS)}/10...")
    m_b = lightgbm.LGBMClassifier(
        objective="binary",
        metric="auc",
        n_estimators=best_iteration_b,
        learning_rate=0.02,
        max_depth=7,
        num_leaves=45,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.75,
        scale_pos_weight=scale_pos_weight_full,
        random_state=seed,
        deterministic=True,
        force_row_wise=True,
        verbose=-1,
    )
    m_b.fit(X_full, y_full)
    test_preds_list.append(m_b.predict_proba(test_df[model_feature_cols])[:, 1])

# 4. Average probabilities across the 10 models to produce final ensemble predictions
test_probs = np.mean(test_preds_list, axis=0)
print("✓ Dual-architecture 10-model ensemble training and prediction complete.")

# 5. Format into the exact competition submission schema (index, target)
submission_df = pd.DataFrame({
    "index": test_df["original_order"],
    "target": test_probs
})

# 6. Automated sanity guardrails before exporting
assert len(submission_df) == len(sample_submission_df), f"Row count mismatch! Expected {len(sample_submission_df)}, got {len(submission_df)}"
assert list(submission_df.columns) == ["index", "target"], f"Column mismatch! Expected ['index', 'target'], got {list(submission_df.columns)}"
assert (submission_df["target"].min() >= 0.0) and (submission_df["target"].max() <= 1.0), "Target probabilities out of [0.0, 1.0] bounds!"
assert submission_df["target"].isna().sum() == 0, "Submission contains NaN predictions!"
assert (submission_df["index"].values == np.arange(len(submission_df))).all(), "Index sequence is non-consecutive or misaligned!"

# 7. Export to submission.csv
SUBMISSION_PATH = "submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"\n✓ Updated submission successfully saved to: {SUBMISSION_PATH}")
print(f"\n--- First 5 rows of submission.csv ---")
print(submission_df.head())
print(f"\n--- Prediction Distribution Summary (Full-Trained Model) ---")
print(submission_df["target"].describe().round(4))

print("STEP 8 COMPLETE ✓")

STEP 8: Retraining on Full Labeled Data & Generating Submission...

Full Labeled Data (Steps 1-35):
  Rows          : 31,235
  Licit (0)     : 27,591
  Illicit (1)   : 3,644
  Scale Weight  : 7.5716
  Trees (rounds): 1142 (optimal round count from Step 6)

Training Dual-Architecture 10-Model Ensemble on steps 1-35 (5 seeds x 2 architectures)...
  [Architecture A - Seed 42] Training model 1/10...
  [Architecture A - Seed 123] Training model 2/10...
  [Architecture A - Seed 456] Training model 3/10...
  [Architecture A - Seed 789] Training model 4/10...
  [Architecture A - Seed 2026] Training model 5/10...
  [Architecture B - Seed 42] Training model 6/10...
  [Architecture B - Seed 123] Training model 7/10...
  [Architecture B - Seed 456] Training model 8/10...
  [Architecture B - Seed 789] Training model 9/10...
  [Architecture B - Seed 2026] Training model 10/10...
✓ Dual-architecture 10-model ensemble training and prediction complete.

✓ Updated submission successfully saved to: submi